# Global Music Trends – Reproducible Analysis

**Project:** GLOBAL MUSIC TRENDS: A data-driven analysis of cultural differences in music streaming  
**Dataset (Kaggle):** https://www.kaggle.com/datasets/anxods/spotify-top-50-playlist-songs-anxods  
**Date:** January 2026

This notebook reproduces the full pipeline:
- Data loading (9 markets + World)
- Data cleaning + standardization
- Data integration (single master dataset)
- Key analyses + visualizations used in the presentation
- Export of the aggregated table for the Datawrapper choropleth map

**Choropleth map (Datawrapper):** https://datawrapper.dwcdn.net/IzK8G/1/


## 1. Environment Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px

from scipy.stats import chi2_contingency

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

sns.set_style("whitegrid")
print("✓ Libraries imported")


## 2. Data Loading

**Expected repository structure**
```
global-music-trends-analysis/
├─ notebooks/
├─ data/
│  └─ raw/   <-- put Kaggle CSV files here
└─ outputs/
   └─ figures/
```


In [ ]:
DATA_DIR = Path("data/raw")
OUTPUT_DIR = Path("outputs")
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

COUNTRY_MAP = {
    "spotify-streaming-top-50-argentina": "Argentina",
    "spotify-streaming-top-50-france": "France",
    "spotify-streaming-top-50-italy": "Italy",
    "spotify-streaming-top-50-japan": "Japan",
    "spotify-streaming-top-50-mexico": "Mexico",
    "spotify-streaming-top-50-south-korea": "South Korea",
    "spotify-streaming-top-50-spain": "Spain",
    "spotify-streaming-top-50-uk": "UK",
    "spotify-streaming-top-50-usa": "USA",
    "spotify-streaming-top-50-world": "World",
}

csv_files = sorted(DATA_DIR.glob("spotify-streaming-top-50-*.csv"))
if len(csv_files) == 0:
    raise FileNotFoundError(
        f"No CSV files found in: {DATA_DIR.resolve()}\n"
        "Download the Kaggle dataset and place all CSV files in: data/raw/"
    )

print(f"Found {len(csv_files)} CSV files:\n")
for fp in csv_files:
    name = fp.stem
    country = COUNTRY_MAP.get(name, name)
    tmp = pd.read_csv(fp)
    print(f"{country:12s} | {tmp.shape[0]:7,} rows | {tmp.shape[1]} columns")


In [ ]:
df_sample = pd.read_csv(csv_files[0])
print("Sample columns:", list(df_sample.columns))
df_sample.head()


## 3. Data Cleaning

Goals:
- Standardize column names
- Ensure correct dtypes (`date`, `position`, `streams`, `duration_ms`, `is_explicit`)
- Handle missing values
- Remove duplicates
- Add `country` tag
- Create `duration_min`


In [ ]:
def _standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )
    return df

def _to_bool(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, bool):
        return x
    s = str(x).strip().lower()
    if s in {"true", "t", "1", "yes", "y"}:
        return True
    if s in {"false", "f", "0", "no", "n"}:
        return False
    return np.nan

def clean_country_df(df_raw: pd.DataFrame, country_name: str) -> pd.DataFrame:
    df = _standardize_columns(df_raw)

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
    if "position" in df.columns:
        df["position"] = pd.to_numeric(df["position"], errors="coerce").astype("Int64")
    if "streams" in df.columns:
        df["streams"] = pd.to_numeric(df["streams"], errors="coerce")
    if "duration_ms" in df.columns:
        df["duration_ms"] = pd.to_numeric(df["duration_ms"], errors="coerce")

    if "is_explicit" in df.columns:
        df["is_explicit"] = df["is_explicit"].map(_to_bool)

    core_cols = [c for c in ["date", "song", "artist", "position"] if c in df.columns]
    before = len(df)
    df = df.dropna(subset=core_cols)
    after = len(df)
    if after != before:
        print(f"{country_name}: dropped {before-after} rows with missing core fields")

    if "is_explicit" in df.columns:
        df["is_explicit"] = df["is_explicit"].fillna(False).astype(bool)

    dedup_cols = [c for c in ["date", "position", "song", "artist"] if c in df.columns]
    before = len(df)
    df = df.drop_duplicates(subset=dedup_cols, keep="first")
    after = len(df)
    if after != before:
        print(f"{country_name}: removed {before-after} duplicate rows")

    df["country"] = country_name

    if "duration_ms" in df.columns:
        df["duration_min"] = df["duration_ms"] / 60000.0

    return df

cleaned = []
for fp in csv_files:
    name = fp.stem
    country = COUNTRY_MAP.get(name, name)
    df_raw = pd.read_csv(fp)
    df_clean = clean_country_df(df_raw, country)
    cleaned.append(df_clean)

print("✓ Cleaning complete")


## 4. Data Integration (Master Dataset)

In [ ]:
df = pd.concat(cleaned, ignore_index=True)
df = df[["country"] + [c for c in df.columns if c != "country"]].copy()

print("Master dataset shape:", df.shape)
print("Countries:", df["country"].nunique())
print("Date range:", df["date"].min().date(), "→", df["date"].max().date())
df.head()


## 5. Descriptive Insights (used in slides)

- Explicit content share by country
- Average song duration by country
- Export of aggregated table for Datawrapper choropleth


In [ ]:
explicit_by_country = (
    df.groupby("country")["is_explicit"]
    .agg(total_tracks="count", explicit_tracks="sum")
    .assign(explicit_pct=lambda x: (x["explicit_tracks"] / x["total_tracks"]) * 100)
    .sort_values("explicit_pct", ascending=False)
)

duration_by_country = (
    df.groupby("country")["duration_min"]
    .mean()
    .sort_values(ascending=False)
    .rename("avg_duration_min")
)

explicit_by_country.head(), duration_by_country.head()


In [ ]:
explicit_export = explicit_by_country.reset_index()[["country", "explicit_pct"]]
explicit_export.to_csv(OUTPUT_DIR / "explicit_share_by_country.csv", index=False)
print("✓ Exported:", OUTPUT_DIR / "explicit_share_by_country.csv")


## 6. Regional Analysis (East vs West)

Regions follow the presentation:
- **Americas:** USA, Mexico, Argentina
- **Europe:** France, Italy, Spain, UK
- **Asia:** Japan, South Korea
- **Global:** World (Spotify global Top 50; not an average of the 9 countries)


In [ ]:
REGIONS = {
    "Americas": ["USA", "Mexico", "Argentina"],
    "Europe": ["France", "Italy", "Spain", "UK"],
    "Asia": ["Japan", "South Korea"],
    "Global": ["World"],
}

country_to_region = {c: r for r, cs in REGIONS.items() for c in cs}
df["region"] = df["country"].map(country_to_region)

regional_explicit = (
    df[df["region"] != "Global"]
    .groupby("region")["is_explicit"]
    .agg(total="count", explicit="sum")
    .assign(explicit_pct=lambda x: (x["explicit"] / x["total"]) * 100)
    .sort_values("explicit_pct", ascending=False)
)

regional_explicit


In [ ]:
cont_table = pd.crosstab(df[df["region"] != "Global"]["region"], df[df["region"] != "Global"]["is_explicit"])
chi2, p, dof, expected = chi2_contingency(cont_table)

print("Chi-square test (Explicit content across regions, excluding Global):")
print(f"chi2={chi2:.4f}, p={p:.4e}, dof={dof}")
print("✓ Significant (p<0.05)" if p < 0.05 else "✗ Not significant (p>=0.05)")


## 7. Visualizations (aligned with slides)

### 7.1 Explicit Content by Country (bar chart)

In [ ]:
plt.figure(figsize=(10, 5))
explicit_by_country["explicit_pct"].sort_values().plot(kind="barh")
plt.xlabel("Explicit content (%)")
plt.title("Explicit content share by country")
plt.tight_layout()
plt.show()


### 7.2 Average Song Duration by Country (bar chart)

In [ ]:
plt.figure(figsize=(10, 5))
duration_by_country.sort_values().plot(kind="barh")
plt.xlabel("Average duration (minutes)")
plt.title("Average song duration by country")
plt.tight_layout()
plt.show()


### 7.3 Heatmap – Explicit Content Distribution Across Chart Positions

**World clarification**  
“World” represents Spotify’s global Top 50 chart, based on aggregated worldwide streaming data.  
It is not an average of the nine analyzed countries.


In [ ]:
countries = sorted(df["country"].unique())
heat = pd.DataFrame(index=range(1, 51), columns=countries, dtype=float)

for c in countries:
    d = df[df["country"] == c]
    for pos in range(1, 51):
        slice_ = d[d["position"] == pos]
        heat.loc[pos, c] = (slice_["is_explicit"].mean() * 100) if len(slice_) else 0.0

plt.figure(figsize=(12, 10))
sns.heatmap(heat, cmap="YlOrRd", vmin=0, vmax=100, cbar_kws={"label": "% Explicit"})
plt.xlabel("Country")
plt.ylabel("Chart position (1 = top)")
plt.title("Explicit content by chart position and country")
plt.tight_layout()
plt.show()


### 7.4 Global vs Local – Cross-border songs

In [ ]:
song_country = (
    df.groupby(["song", "artist"])["country"]
    .nunique()
    .sort_values(ascending=False)
)

max_countries = df["country"].nunique()
print("Max countries in dataset:", max_countries)
print("\nTop songs by number of countries:")
print(song_country.head(10))

print("\nSongs in all 9 countries:", (song_country == 9).sum())
print("Songs in 7+ countries:", (song_country >= 7).sum())
print("Unique songs analyzed:", df[["song", "artist"]].drop_duplicates().shape[0])


## 8. Choropleth Map (Datawrapper)

The choropleth map (explicit content share by country) was created using Datawrapper from the aggregated table produced in this notebook.

**Interactive map:** https://datawrapper.dwcdn.net/IzK8G/1/

**Replication steps:**
1. Run this notebook and generate `outputs/explicit_share_by_country.csv`
2. Upload the CSV to Datawrapper (https://app.datawrapper.de/)
3. Choose **Choropleth Map**
4. Match country names/codes and publish


## 9. Reproducibility Notes

**How to run**
1. Download the dataset from Kaggle: https://www.kaggle.com/datasets/anxods/spotify-top-50-playlist-songs-anxods
2. Place all CSV files in `data/raw/`
3. Install dependencies:
   ```bash
   pip install -r requirements.txt
   ```
4. Run the notebook top-to-bottom.

**Data note**
The raw dataset is not included in the repository (licensing/size). The analysis is fully reproducible using the Kaggle dataset above.
